In [8]:
# Copyright (c) 2022-2025, The Isaac Lab Project Developers.
# All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

![Isaac Lab](../../../docs/source/_static/isaaclab.jpg)

---

# Isaac Lab

**Isaac Lab** is a GPU-accelerated, open-source framework designed to unify and simplify robotics research workflows, such as reinforcement learning, imitation learning, and motion planning. Built on [NVIDIA Isaac Sim](https://docs.isaacsim.omniverse.nvidia.com/latest/index.html), it combines fast and accurate physics and sensor simulation, making it an ideal choice for sim-to-real transfer in robotics.

Isaac Lab provides developers with a range of essential features for accurate sensor simulation, such as RTX-based cameras, LIDAR, or contact sensors. The framework's GPU acceleration enables users to run complex simulations and computations faster, which is key for iterative processes like reinforcement learning and data-intensive tasks. Moreover, Isaac Lab can run locally or be distributed across the cloud, offering flexibility for large-scale deployments.

## Key Features

Isaac Lab offers a comprehensive set of tools and environments designed to facilitate robot learning:
- **Robots**: A diverse collection of robots, from manipulators, quadrupeds, to humanoids, with 16 commonly available models.
- **Environments**: Ready-to-train implementations of more than 30 environments, which can be trained with popular reinforcement learning frameworks such as RSL RL, SKRL, RL Games, or Stable Baselines. We also support multi-agent reinforcement learning.
- **Physics**: Rigid bodies, articulated systems, deformable objects
- **Sensors**: RGB/depth/segmentation cameras, camera annotations, IMU, contact sensors, ray casters.

# Outline
The notebook is structured to guide users through the complete process of setting up and running reinforcement learning experiments using the Isaac Lab framework, with detailed configurations for PPO training on robotics tasks.

1. Introduction
2. Setup and Imports
3. Configuration settings
4. Training Setup
5. Training Execution
6. PPO Configuration Details
7. Environment Configuration

In [9]:
"""Script to train RL agent with RSL-RL.

This script provides functionality to train reinforcement learning agents using RSL-RL.
Must launch Isaac Sim Simulator before running this script.
"""

# standard libraries
import sys
from typing import Any

# isaaclab specific
import cli_args
from isaaclab.app import AppLauncher


| Argument | Default Value | Description |
|----------|---------------|-------------|
| --video | False | Record videos during training |
| --video_length | 200 | Length of recorded video (in steps) |
| --video_interval | 2000 | Interval between video recordings (in steps) |
| --num_envs | 20 | Number of environments to simulate |
| --task | "Isaac-Velocity-Rough-H1-v0" | Name of the task |
| --seed | 42 | Seed used for the environment |
| --max_iterations | 10000 | RL Policy training iterations |
| --headless | False | Force display off at all times |
| --livestream | -1 | Force enable livestreaming (0,1,2) |
| --enable_cameras | False | Enable camera sensors and dependencies |
| --device | "cuda:0" | Device to run simulation on |
| --verbose | False | Enable verbose-level SimulationApp logs |
| --info | False | Enable info-level SimulationApp logs |
| --experience | "" | Experience file to load for SimulationApp |
| --kit_args | "" | Command line arguments for Omniverse Kit |

In [10]:
import argparse

_APPLAUNCHER_CFG_INFO: dict[str, tuple[list[type], Any]] = {
"headless": ([bool], False),
"livestream": ([int], -1),
"enable_cameras": ([bool], False),
"device": ([str], "cuda:0"),
"experience": ([str], ""),
}

# add argparse arguments
parser = argparse.ArgumentParser(description="Train an RL agent with RSL-RL.")
parser.add_argument("--video", action="store_true", default=False, help="Record videos during training.")
parser.add_argument("--video_length", type=int, default=200, help="Length of the recorded video (in steps).")
parser.add_argument("--video_interval", type=int, default=2000, help="Interval between video recordings (in steps).")
parser.add_argument("--num_envs", type=int, default=10, help="Number of environments to simulate.")
parser.add_argument("--task", type=str, default="Isaac-Velocity-Rough-H1-v0", help="Name of the task.")
parser.add_argument("--seed", type=int, default=42, help="Seed used for the environment")
parser.add_argument("--max_iterations", type=int, default=10000, help="RL Policy training iterations.")
arg_group = parser.add_argument_group(
    "app_launcher arguments",
    description="Arguments for the AppLauncher. For more details, please check the documentation.",
)
arg_group.add_argument(
    "--headless",
    action="store_true",
    default=AppLauncher._APPLAUNCHER_CFG_INFO["headless"][1],
    help="Force display off at all times.",
)
arg_group.add_argument(
    "--livestream",
    type=int,
    default=AppLauncher._APPLAUNCHER_CFG_INFO["livestream"][1],
    choices={0, 1, 2},
    help="Force enable livestreaming. Mapping corresponds to that for the `LIVESTREAM` environment variable.",
)
arg_group.add_argument(
    "--enable_cameras",
    action="store_true",
    default=AppLauncher._APPLAUNCHER_CFG_INFO["enable_cameras"][1],
    help="Enable camera sensors and relevant extension dependencies.",
)
arg_group.add_argument(
    "--device",
    type=str,
    default=AppLauncher._APPLAUNCHER_CFG_INFO["device"][1],
    help='The device to run the simulation on. Can be "cpu", "cuda", "cuda:N", where N is the device ID',
)
# Add the deprecated cpu flag to raise an error if it is used
arg_group.add_argument("--cpu", action="store_true", help=argparse.SUPPRESS)
arg_group.add_argument(
    "--verbose",  # Note: This is read by SimulationApp through sys.argv
    action="store_true",
    help="Enable verbose-level log output from the SimulationApp.",
)
arg_group.add_argument(
    "--info",  # Note: This is read by SimulationApp through sys.argv
    action="store_true",
    help="Enable info-level log output from the SimulationApp.",
)
arg_group.add_argument(
    "--experience",
    type=str,
    default="",
    help=(
        "The experience file to load when launching the SimulationApp. If an empty string is provided,"
        " the experience file is determined based on the headless flag. If a relative path is provided,"
        " it is resolved relative to the `apps` folder in Isaac Sim and Isaac Lab (in that order)."
    ),
)
arg_group.add_argument(
    "--kit_args",
    type=str,
    default="",
    help=(
        "Command line arguments for Omniverse Kit as a string separated by a space delimiter."
        ' Example usage: --kit_args "--ext-folder=/path/to/ext1 --ext-folder=/path/to/ext2"'
    ),
)

# append RSL-RL cli arguments
cli_args.add_rsl_rl_args(parser)

# append AppLauncher cli args
args_cli, hydra_args = parser.parse_known_args()
# always enable cameras to record video
if args_cli.video:
    args_cli.enable_cameras = True

# clear out sys.argv for Hydra
sys.argv = [sys.argv[0]] + hydra_args
args_cli.headless = False

In [11]:
# Display the parsed command line arguments
args_cli.__dict__

{'video': False,
 'video_length': 200,
 'video_interval': 2000,
 'num_envs': 10,
 'task': 'Isaac-Velocity-Rough-H1-v0',
 'seed': 42,
 'max_iterations': 10000,
 'headless': False,
 'livestream': -1,
 'enable_cameras': False,
 'device': 'cuda:0',
 'cpu': False,
 'verbose': False,
 'info': False,
 'experience': '',
 'kit_args': '',
 'experiment_name': None,
 'run_name': None,
 'resume': None,
 'load_run': None,
 'checkpoint': None,
 'logger': None,
 'log_project_name': None}

### Training H1 Humanoid Robot
![H1 Training](../../../docs/source/_static/tasks/locomotion/h1_train.gif)

In [12]:
# # Note: The following is only needed when running in Jupyter notebooks.
# # When running as a Python script, this is not required.
# # See: scripts/reinforcement_learning/rsl_rl/train.py
import nest_asyncio
nest_asyncio.apply()

In [13]:
# # launch omniverse app
app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app

[INFO][AppLauncher]: Loading experience file: /home/ksachdev/IsaacLab/apps/isaaclab.python.kit
[Warning] [simulation_app.simulation_app] Modules: ['omni.kit_app'] were loaded before SimulationApp was started and might not be loaded correctly.
[Warning] [simulation_app.simulation_app] Please check to make sure no extra omniverse or pxr modules are imported before the call to SimulationApp(...)
Loading user config located at: '/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/omni/data/Kit/Isaac-Sim/4.5/user.config.json'
[Info] [carb] Logging to file: /home/ksachdev/env_isaaclab/lib/python3.10/site-packages/omni/logs/Kit/Isaac-Sim/4.5/kit_20250218_134325.log
2025-02-18 12:43:25 s] [Warning] [omni.kit.app.plugin] No crash reporter present, dumps uploading isn't available.
[0.050s] [ext: omni.kit.async_engine-0.0.1] startup
[0.159s] [ext: omni.metrics.core-0.0.1] startup
[0.160s] [ext: omni.client.lib-1.0.0] startup
[0.186s] [ext: omni.blobkey-1.1.2] startup
[0.186s] [ext: omni.stat

In [14]:
# Standard imports for system, ML and timing
import gymnasium as gym
import os
import torch
from datetime import datetime

# RSL-RL specific imports for RL training
from isaaclab_rl.rsl_rl import RslRlOnPolicyRunnerCfg, RslRlVecEnvWrapper
from rsl_rl.runners import OnPolicyRunner

# Environment related imports from isaaclab
from isaaclab.envs import (
    DirectMARLEnv,
    DirectMARLEnvCfg,
    DirectRLEnvCfg,
    ManagerBasedRLEnvCfg,
    multi_agent_to_single_agent,
)
from isaaclab.utils.dict import print_dict
from isaaclab.utils.io import dump_pickle, dump_yaml

# Task specific imports and utilities
import isaaclab_tasks
from isaaclab_tasks.utils import get_checkpoint_path
from isaaclab_tasks.utils.hydra import register_task_to_hydra

# PyTorch performance optimization settings
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False

# Set __file__ to current working directory for notebook compatibility
__file__ = os.getcwd()

In [15]:
import yaml
from typing import Any
from isaaclab.utils import class_to_dict

def represent_tuple(dumper: yaml.Dumper, data: tuple) -> yaml.Node:
    """Convert Python tuple to YAML sequence."""
    return dumper.represent_sequence('tag:yaml.org,2002:seq', list(data))

def represent_slice(dumper: yaml.Dumper, data: slice) -> yaml.Node:
    """Convert Python slice to YAML null if all attributes are None."""
    if data.start is None and data.stop is None and data.step is None:
        return dumper.represent_scalar('tag:yaml.org,2002:null', '')
    # If not all None, represent as sequence of start:stop:step
    return dumper.represent_sequence('tag:yaml.org,2002:seq', 
                                   [data.start, data.stop, data.step])

def dump_yaml(filename: str, data: dict | object, sort_keys: bool = False):
    """Saves data into a YAML file safely with improved readability.

    Note:
        The function creates any missing directory along the file's path.
        Handles Python objects like tuples and slices in a more readable format.

    Args:
        filename: The path to save the file at.
        data: The data to save either a dictionary or class object.
        sort_keys: Whether to sort the keys in the output file. Defaults to False.
    """
    # check ending
    if not filename.endswith("yaml"):
        filename += ".yaml"
    # create directory
    if not os.path.exists(os.path.dirname(filename)):
        os.makedirs(os.path.dirname(filename), exist_ok=True)
    # convert data into dictionary
    if not isinstance(data, dict):
        data = class_to_dict(data)

    # Create custom dumper with better representations
    class CustomDumper(yaml.SafeDumper):
        pass
    
    # Register custom representers
    CustomDumper.add_representer(tuple, represent_tuple)
    CustomDumper.add_representer(slice, represent_slice)
    CustomDumper.ignore_aliases = lambda *args: True  # Prevent alias references
    
    # save data
    with open(filename, "w") as f:
        yaml.dump(data, f, 
                 Dumper=CustomDumper,
                 default_flow_style=False, 
                 sort_keys=sort_keys)

[14.585s] Simulation App Startup Complete
[INFO]: Parsing configuration from: isaaclab_tasks.manager_based.locomotion.velocity.config.h1.rough_env_cfg:H1RoughEnvCfg
[INFO]: Parsing configuration from: isaaclab_tasks.manager_based.locomotion.velocity.config.h1.agents.rsl_rl_ppo_cfg:H1RoughPPORunnerCfg
[INFO] Logging experiment in directory: /home/ksachdev/IsaacLab/scripts/reinforcement_learning/rsl_rl/logs/rsl_rl/h1_rough
Exact experiment name requested from command line: 2025-02-18_13-43-40
Setting seed: 42
[14.961s] [ext: omni.physx.fabric-106.5.7] startup
[INFO]: Base environment:
	Environment device    : cuda:0
	Environment seed      : 42
	Physics step-size     : 0.005
	Rendering step-size   : 0.02
	Environment step-size : 0.02
[INFO] Generating terrains based on curriculum took : 1.250858 seconds
[INFO]: Time taken for scene creation : 3.127011 seconds
[INFO]: Scene manager:  <class InteractiveScene>
	Number of environments: 10
	Environment spacing   : 2.5
	Source prim name      : 

In [16]:
def main(env_cfg: ManagerBasedRLEnvCfg | DirectRLEnvCfg | DirectMARLEnvCfg, agent_cfg: RslRlOnPolicyRunnerCfg):
    """Train with RSL-RL agent."""
    # override configurations with non-hydra CLI arguments
    agent_cfg = cli_args.update_rsl_rl_cfg(agent_cfg, args_cli)
    env_cfg.scene.num_envs = args_cli.num_envs if args_cli.num_envs is not None else env_cfg.scene.num_envs
    agent_cfg.max_iterations = (
        args_cli.max_iterations if args_cli.max_iterations is not None else agent_cfg.max_iterations
    )

    # set the environment seed
    # note: certain randomizations occur in the environment initialization so we set the seed here
    env_cfg.seed = agent_cfg.seed
    
    env_cfg.sim.device = args_cli.device if args_cli.device is not None else env_cfg.sim.device

    # specify directory for logging experiments
    log_root_path = os.path.join("logs", "rsl_rl", agent_cfg.experiment_name)
    log_root_path = os.path.abspath(log_root_path)
    print(f"[INFO] Logging experiment in directory: {log_root_path}")
    
    # specify directory for logging runs: {time-stamp}_{run_name}
    log_dir = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    # This way, the Ray Tune workflow can extract experiment name.
    print(f"Exact experiment name requested from command line: {log_dir}")
    if agent_cfg.run_name:
        log_dir += f"_{agent_cfg.run_name}"
    log_dir = os.path.join(log_root_path, log_dir)

    # create isaac environment
    env = gym.make(args_cli.task, cfg=env_cfg, render_mode="rgb_array" if args_cli.video else None)

    # convert to single-agent instance if required by the RL algorithm
    if isinstance(env.unwrapped, DirectMARLEnv):
        env = multi_agent_to_single_agent(env)

    # save resume path before creating a new log_dir
    if agent_cfg.resume:
        resume_path = get_checkpoint_path(log_root_path, agent_cfg.load_run, agent_cfg.load_checkpoint)

    # wrap for video recording
    if args_cli.video:
        video_kwargs = {
            "video_folder": os.path.join(log_dir, "videos", "train"),
            "step_trigger": lambda step: step % args_cli.video_interval == 0,
            "video_length": args_cli.video_length,
            "disable_logger": True,
        }
        print("[INFO] Recording videos during training.")
        print_dict(video_kwargs, nesting=4)
        env = gym.wrappers.RecordVideo(env, **video_kwargs)

    # wrap around environment for rsl-rl
    env = RslRlVecEnvWrapper(env)

    # create runner from rsl-rl
    runner = OnPolicyRunner(env, agent_cfg.to_dict(), log_dir=log_dir, device=agent_cfg.device)
    # write git state to logs
    runner.add_git_repo_to_log(__file__)
    # load the checkpoint
    if agent_cfg.resume:
        print(f"[INFO]: Loading model checkpoint from: {resume_path}")
        # load previously trained model
        runner.load(resume_path)

    # dump the configuration into log-directory
    dump_yaml(os.path.join(log_dir, "params", "env.yaml"), env_cfg)
    dump_yaml(os.path.join(log_dir, "params", "agent.yaml"), agent_cfg)
    dump_pickle(os.path.join(log_dir, "params", "env.pkl"), env_cfg)
    dump_pickle(os.path.join(log_dir, "params", "agent.pkl"), agent_cfg)

    # run training
    runner.learn(num_learning_iterations=agent_cfg.max_iterations, init_at_random_ep_len=True)

    # close the simulator
    env.close()

In [ ]:
# Register task configuration with Hydra and get environment and agent configs
env_cfg, agent_cfg = register_task_to_hydra(args_cli.task, "rsl_rl_cfg_entry_point")

# Run training loop
main(env_cfg, agent_cfg)

/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment Isaac-Repose-Cube-Allegro-Direct-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment Isaac-Ant-Direct-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment Isaac-Velocity-Flat-Anymal-C-Direct-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/gymnasium/envs/registration.py:642: UserWarning: WARN: Overriding environment Isaac-Velocity-Rough-Anymal-C-Direct-v0 already in registry.
  logger.warn

In [ ]:
# close sim app
simulation_app.close()

/home/ksachdev/IsaacLab/source/isaaclab/isaaclab/envs/manager_based_rl_env.py

source/isaaclab_tasks/isaaclab_tasks/manager_based/locomotion/velocity/config/h1/agents/rsl_rl_ppo_cfg.py
# PPO Configuration - H1 Robot Rough Terrain Training

## Training Loop
* 24 steps per environment before update
* 3000 total training iterations
* Checkpoints saved every 50 iterations
* Experiment name: "h1_rough"
* No empirical normalization

## Neural Networks
* Dual network architecture:
  * Actor network for action selection
  * Critic network for value estimation
* Hidden layers: [512, 256, 128] neurons
* ELU activation functions
* Initial exploration noise: 1.0

## PPO Parameters
* Learning rate: 0.001 (adaptive)
* Training:
  * 5 epochs per batch
  * 4 mini-batches per update
* Loss functions:
  * Clipped value loss
  * Clipped PPO objective
* Hyperparameters:
  * Discount (γ): 0.99
  * GAE (λ): 0.95
  * Entropy coefficient: 0.01
  * Gradient clip: 1.0
  * Target KL: 0.01

# Adaptive Learning Rate Scheduler
 
## Overview
* Dynamically adjusts learning rate based on policy change magnitude
* Uses KL divergence to measure difference between old and new policies
* Visual explanation: https://spinningup.openai.com/en/latest/_images/kl_divergence.svg

## How it works
* Measures KL divergence between policy distributions:
  * High KL = Policy changing too fast
  * Low KL = Policy changing too slowly
  * Target KL = 0.01 (desired amount of change)

## Adjustment Rules
* If KL > 2× target:
  * Policy changing too fast
  * Decrease learning rate by 1.5×
  * Min learning rate = 1e-5
* If KL < 0.5× target:
  * Policy changing too slowly  
  * Increase learning rate by 1.5×
  * Max learning rate = 1e-2
  
  
  ```python
    # adaptive learning rate scheduler
    if self.desired_kl is not None and self.schedule == "adaptive":
        with torch.inference_mode():
            kl = torch.sum(
                torch.log(sigma_batch / old_sigma_batch + 1.0e-5)
                + (torch.square(old_sigma_batch) + torch.square(old_mu_batch - mu_batch))
                / (2.0 * torch.square(sigma_batch))
                - 0.5,
                axis=-1,
            )
            kl_mean = torch.mean(kl)

            if kl_mean > self.desired_kl * 2.0:
                self.learning_rate = max(1e-5, self.learning_rate / 1.5)
            elif kl_mean < self.desired_kl / 2.0 and kl_mean > 0.0:
                self.learning_rate = min(1e-2, self.learning_rate * 1.5)

            for param_group in self.optimizer.param_groups:
                param_group["lr"] = self.learning_rate
  ```